In [1]:
# --- Celda 1.1: Importaciones y Carga de Librerías Offline ---
import os
import numpy as np
import pandas as pd
import glob
import ROOT
import time
import traceback
from multiprocessing import Pool, cpu_count # <-- Importamos Pool

# Esta es la parte MÁS IMPORTANTE:
# Asegúrate de que estás corriendo este Jupyter Lab desde una terminal
# donde ANTES hiciste: source /ruta/a/auger/offline/this-auger-offline.sh

AugerOfflineRoot = os.environ.get("AUGEROFFLINEROOT")
if AugerOfflineRoot is None:
    raise EnvironmentError(
        "AUGEROFFLINEROOT no definido. "
        "Reinicia Jupyter Lab desde una terminal donde hayas "
        "hecho: "
        " 'aug_set_version offline 4.0.1-icrc23-prod1-root6' "   
        " 'source /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6/bin/this-auger-offline.sh'."
    )

print(f"AUGEROFFLINEROOT encontrado en: {AugerOfflineRoot}")

# Cargar las librerías necesarias
print("Cargando librerías de Auger Offline...")
libs_to_load = ["libRecEventKG.so"]
for lib in libs_to_load:
    lib_path = os.path.join(AugerOfflineRoot, "lib", lib)
    if not os.path.exists(lib_path):
        raise FileNotFoundError(f"No se encontró la librería: {lib_path}")
    
    # Usamos gSystem.Load que es más robusto en PyROOT
    status = ROOT.gSystem.Load(lib_path)
    if status < 0:
        raise ImportError(f"Error cargando la librería: {lib_path}")

print("Librerías cargadas correctamente. ¡Listo para trabajar! 🚀")

Welcome to JupyROOT 6.30/04
AUGEROFFLINEROOT encontrado en: /srv/software/amd64/ubuntu/24.04/auger/offline/4.0.1-icrc23-prod1-root6
Cargando librerías de Auger Offline...
Librerías cargadas correctamente. ¡Listo para trabajar! 🚀


In [2]:
# --- Celda 2.1: Funciones Auxiliares y Principales (Actualizada) ---

def getCounterList(sevent, mevent):
    """
    Obtener la lista de counters UMD asociados a las estaciones SD de un evento.
    """
    cList = []
    stationVector = sevent.GetStationVector()
    for station in stationVector:
        stationId = station.GetId()
        counterId = int("10" + str(stationId))
        if mevent.HasCounter(counterId):
            cList.append(mevent.GetCounter(counterId))
    return cList

def getModuleList(counter, sim=True):
    """
    Obtener la lista de módulos (scintillators) de un counter UMD.
    """
    possibleModules = range(0, 6) if sim else range(100, 116)
    modules = []
    for modId in possibleModules:
        if counter.HasModule(modId):
            modules.append(counter.GetModule(modId))
    return modules

def readADST_surface(fname):
    """
    Leer un archivo ADST y extraer info en un DataFrame "plano".
    Cada fila corresponde a UN counter UMD.
    """
    
    print(f"Iniciando lectura de: {os.path.basename(fname)}")
    
    if not os.path.exists(fname):
        print(f"Advertencia: Archivo no encontrado {fname}")
        return pd.DataFrame() # Retorna DF vacío

    files = ROOT.std.vector('string')()
    files.push_back(fname)

    # Inicialización ADST
    file1 = ROOT.RecEventFile(files)
    event = ROOT.RecEvent()
    geo = ROOT.DetectorGeometry()
    
    # Leemos la geometría. Ignoramos si falla.
    file1.ReadDetectorGeometry(geo)
    file1.SetBuffers(event)

    data = [] # Esta lista contendrá una fila por COUNTER
    event_count = 0
    start_time = time.time()

    while file1.ReadNextEvent() == ROOT.RecEventFile.eSuccess:
        event_count += 1
        if event_count % 500 == 0:
            print(f"... procesados {event_count} eventos.")

        # --- ❗️❗️❗️ NUEVA LÍNEA ❗️❗️❗️ ---
        # Capturamos el ID único del evento (lluvia)
        event_id_lluvia = event.GetEventId()

        # ----------------- MC -----------------
        MCShower = event.GetGenShower()
        logE_MC = np.log10(MCShower.GetEnergy())
        theta_MC = MCShower.GetZenith() * 180.0 / np.pi
        phi_MC = MCShower.GetAzimuth() * 180.0 / np.pi
        primary = MCShower.GetShortPrimaryName()
        
        # ----------------- REC -----------------
        sEvent = event.GetSDEvent()
        sShower = sEvent.GetSdRecShower()
        logE_REC = np.log10(sShower.GetEnergy())
        theta_REC = sShower.GetZenith() * 180.0 / np.pi
        phi_REC = sShower.GetAzimuth() * 180.0 / np.pi
        
        # ----------------- MD -----------------
        mEvent = event.GetMDEvent()
        counterList = getCounterList(sEvent, mEvent)

        if not counterList:
            continue # Saltamos eventos sin señal UMD

        for counter in counterList:
            if counter.IsRejected() or counter.IsSaturated():
                print('skipped counter')
                continue

            # Número total de muones por counter            
            nMuones = counter.GetNumberOfMuons()

            # Coordenadas SD asociadas
            sdId = counter.GetSdPartnerId()
            sdStation = sEvent.GetStationById(sdId) if sEvent.HasStation(sdId) else None
            if sdStation:
                r = sdStation.GetSPDistance()
                phi_rel = sdStation.GetAzimuthSP() - sShower.GetAzimuth()
                x_plane = r * np.cos(phi_rel)
                y_plane = r * np.sin(phi_rel)
                r_core = r
                sdSignal = sdStation.GetTotalSignal()
                # Calculamos el phi en el plano de la lluvia (0 a 2*pi)
                phi_plane = np.arctan2(y_plane, x_plane) % (2 * np.pi) # <-- AÑADIDO
            else:
                x_plane, y_plane, r_core, sdSignal, phi_plane = None, None, None, None, None # <-- AÑADIDO

            data.append({
                "event_id": event_id_lluvia, # El ID único de la lluvia
            
                # Info MC
                "logE_MC": logE_MC, "theta_MC": theta_MC, "phi_MC": phi_MC, "primary": primary,
                
                # Info REC
                "logE_REC": logE_REC, "theta_REC": theta_REC, "phi_REC": phi_REC,
                
                # Info específica del Counter
                "counterId": counter.GetId(),
                "nMuones": nMuones,
                "x_plane": x_plane,
                "y_plane": y_plane,
                "phi_plane": phi_plane, # <-- AÑADIDO
                "r_core": r_core,
                "sdId": sdId,
                "sdSignal": sdSignal
            })

    end_time = time.time()
    elapsed = end_time - start_time
    print(f"Lectura completa. Total de eventos leídos: {event_count}")
    print(f"Tiempo total de lectura: {elapsed:.2f} segundos.")
    print(f"Total de 'counters' (filas) extraídos: {len(data)}")

    df = pd.DataFrame(data)
    return df

# Prueba de Procesamiento de los archivos

In [10]:
print("--- INICIANDO PROCESO SERIAL (MODO DE PRUEBA: 1 ARCHIVO) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/Prueba_ADST_Alexey/parquet/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

# --- 3. Bucle de Procesamiento Serial ---
exitos = 0
errores = 0

for i, root_fpath in enumerate(all_root_files):
    
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"\n--- Procesando archivo {i+1}/{len(all_root_files)} ---")
    print(f"► {filename}")
    
    # Evita reprocesar archivos que ya existen
    if os.path.exists(output_path):
        print(f"INFO: El archivo ya existe, saltando: {output_filename}")
        continue

    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (usa la Celda 2.1)
        df = readADST_surface(root_fpath)
        
        if df.empty:
            print(f"INFO: Archivo vacío o sin datos UMD. Saltando.")
            continue

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        print(f"✔ [Éxito]: Guardado como {output_filename}")
        print(f"Tiempo de este archivo: {end_file_time - start_file_time:.2f}s")
        exitos += 1
        
        # ----- FIN DEL TRABAJO -----
        
        # ❗️ LÍNEA DE PRUEBA AÑADIDA ❗️
        print("\n--- ¡PRUEBA DETENIDA! Saliendo del bucle después de 1 archivo. ---")
        break  # Esto detiene el 'for' después de la primera pasada exitosa
               # ELIMINÁ ESTA LÍNEA para procesar todos los archivos.

    except Exception as e:
        print(f"❌ [ERROR] en {filename}: {e}")
        print(traceback.format_exc())
        errores += 1

# --- 4. Resumen Final ---
end_total_time = time.time()
print("\n\n--- Proceso Completado ---")
print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
print(f"Total de archivos procesados: {exitos + errores}")
print(f"Éxitos: {exitos}")
print(f"Errores: {errores}")
print(f"¡Listo! Tu archivo .parquet de prueba está en: {output_dir}")

--- INICIANDO PROCESO SERIAL (MODO DE PRUEBA: 1 ARCHIVO) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Ruso/parquet/
Encontrados 20 archivos .root para procesar.

--- Procesando archivo 1/20 ---
► SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root
Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root


/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())


... procesados 500 eventos.
... procesados 1000 eventos.
Lectura completa. Total de eventos leídos: 1223
Tiempo total de lectura: 402.30 segundos.
Total de 'counters' (filas) extraídos: 12729
✔ [Éxito]: Guardado como SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.parquet
Tiempo de este archivo: 402.77s

--- ¡PRUEBA DETENIDA! Saliendo del bucle después de 1 archivo. ---


--- Proceso Completado ---
Tiempo total: 6.71 minutos
Total de archivos procesados: 1
Éxitos: 1
Errores: 0
¡Listo! Tu archivo .parquet de prueba está en: /home/lsilva/Github/ADST_Ruso/parquet/


  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.root


# Paralelizacion (4 instancias)

In [14]:
# -----------------------------------------------------------------
# FUNCIÓN "TRABAJADORA"
# Esta función contiene toda la lógica que antes estaba en tu bucle.
# Se ejecutará en un proceso separado para CADA archivo.
# -----------------------------------------------------------------
def process_file_wrapper(root_fpath):
    
    # 1. Definir rutas
    output_dir = "/home/lsilva/Github/ADST_Alexey/parquet/" # Ruta de salida
    filename = os.path.basename(root_fpath)
    output_filename = filename.replace(".root", ".parquet")
    output_path = os.path.join(output_dir, output_filename)
    
    # 2. Evitar reprocesar
    if os.path.exists(output_path):
        return f"INFO: El archivo ya existe, saltando: {output_filename}"

    # 3. Imprimir estado
    # (En paralelo, los prints pueden salir desordenados, es normal)
    print(f"► [Iniciando]: {filename}")
    
    try:
        # ----- INICIO DEL TRABAJO -----
        start_file_time = time.time()
        
        # 1. Leer el .root (¡Llama a la función de la Celda 2.1!)
        # (Asegurate de haber ejecutado tu Celda 2.1 con la
        # función readADST_surface() antes de correr esta celda)
        df = readADST_surface(root_fpath)
        
        if df.empty:
            return f"INFO: Archivo vacío o sin datos UMD. Saltando: {filename}"

        # 2. Extraer metadatos del nombre de archivo
        try:
            parts = filename.split('_')
            df["model_mc"] = parts[0]
            df["e_min_mc"] = float(parts[1]) / 10.0
            df["e_max_mc"] = float(parts[2]) / 10.0
            df["primary_name_mc"] = parts[3]
            run_part = parts[-1].replace('.root', '')
            df["run_number"] = int(run_part.replace('Run', ''))
        except Exception as e_parse:
            print(f"  Advertencia: No se pudo parsear metadata en {filename}: {e_parse}")

        # 3. Guardar en Parquet
        df.to_parquet(
            output_path,
            compression="snappy",
            index=False
        )
        
        # 4. Liberar memoria
        del df
        
        end_file_time = time.time()
        elapsed = end_file_time - start_file_time
        return f"✔ [Éxito]: {filename} -> {output_filename} ({elapsed:.2f}s)"

        # ----- FIN DEL TRABAJO -----

    except Exception as e:
        # Si algo falla, retornamos el string de error
        return f"❌ [ERROR] en {filename}: {e}\n{traceback.format_exc()}"

In [15]:
# -----------------------------------------------------------------
# SCRIPT PRINCIPAL
# -----------------------------------------------------------------

print("--- INICIANDO PROCESO PARALELO (4 Workers) ---")
start_total_time = time.time()

# --- 1. Configuración de Rutas ---
base_path = "/srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium"
output_dir = "/home/lsilva/Github/ADST_Ruso/parquet/"

os.makedirs(output_dir, exist_ok=True)
print(f"Buscando archivos en: {base_path}")
print(f"Los archivos .parquet se guardarán en: {output_dir}")

# --- 2. Encontrar todos los archivos ---
all_root_files = glob.glob(os.path.join(base_path, "*.root"))
all_root_files.sort()

if not all_root_files:
    print(f"¡Error! No se encontraron archivos .root en: {base_path}")
else:
    print(f"Encontrados {len(all_root_files)} archivos .root para procesar.")

    # --- 3. Bucle de Procesamiento Paralelo ---
    n_workers = 4  # Ajustar como sea necesario
    print(f"Iniciando Pool con {n_workers} trabajadores...")


    with Pool(processes=n_workers) as pool:
        
        # pool.map() distribuye la lista 'all_root_files'
        # entre los 4 trabajadores y aplica la función 'process_file_wrapper'
        # 'results' será una lista con los strings de "Éxito" o "ERROR"
        results = pool.map(process_file_wrapper, all_root_files)
    
    print("\n\n--- Proceso Paralelo Completado ---")

    # --- 4. Resumen Final ---
    exitos = 0
    errores = 0
    
    # Imprimimos todos los mensajes de resultado
    for res in results:
        print(res)
        if "✔ [Éxito]" in res:
            exitos += 1
        elif "❌ [ERROR]" in res:
            errores += 1

    end_total_time = time.time()
    print("\n--- Resumen de la Tanda ---")
    print(f"Tiempo total: {(end_total_time - start_total_time) / 60:.2f} minutos")
    print(f"Total de archivos: {len(all_root_files)}")
    print(f"Éxitos: {exitos}")
    print(f"Errores: {errores}")
    print(f"¡Listo! Tus archivos .parquet están en: {output_dir}")

--- INICIANDO PROCESO PARALELO (4 Workers) ---
Buscando archivos en: /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium
Los archivos .parquet se guardarán en: /home/lsilva/Github/ADST_Ruso/parquet/
Encontrados 20 archivos .root para procesar.
Iniciando Pool con 4 trabajadores...
► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run011.root



Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.rootIniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.rootIniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.rootIniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run011.root





/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())
/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())
/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())
/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())


... procesados 500 eventos.
... procesados 500 eventos.
... procesados 500 eventos.
... procesados 500 eventos.
... procesados 1000 eventos.
... procesados 1000 eventos.
... procesados 1000 eventos.
... procesados 1000 eventos.
Lectura completa. Total de eventos leídos: 1217
Tiempo total de lectura: 377.23 segundos.
Total de 'counters' (filas) extraídos: 12888
► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run032.root
Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run032.root
Lectura completa. Total de eventos leídos: 1219
Tiempo total de lectura: 377.84 segundos.
Total de 'counters' (filas) extraídos: 12758


/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())


► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run030.root
Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run030.root


...

/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())


Lectura completa. Total de eventos leídos: 1213
Tiempo total de lectura: 362.65 segundos.
Total de 'counters' (filas) extraídos: 12509
Lectura completa. Total de eventos leídos: 1218
Tiempo total de lectura: 359.84 segundos.
Total de 'counters' (filas) extraídos: 12314
... procesados 500 eventos.
... procesados 500 eventos.
... procesados 1000 eventos.
... procesados 1000 eventos.
Lectura completa. Total de eventos leídos: 1213
Tiempo total de lectura: 357.20 segundos.
Total de 'counters' (filas) extraídos: 12333
► [Iniciando]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run094.root
Iniciando lectura de: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run094.root
Lectura completa. Total de eventos leídos: 1209
Tiempo total de lectura: 355.97 segundos.
Total de 'counters' (filas) extraídos: 12238


/tmp/ipykernel_3470253/4173320093.py:74: RuntimeWarning: divide by zero encountered in log10
  logE_REC = np.log10(sShower.GetEnergy())


... procesados 500 eventos.
... procesados 1000 eventos.
Lectura completa. Total de eventos leídos: 1220
Tiempo total de lectura: 386.50 segundos.
Total de 'counters' (filas) extraídos: 12450


--- Proceso Paralelo Completado ---
INFO: El archivo ya existe, saltando: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run010.parquet
✔ [Éxito]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run011.root -> SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run011.parquet (383.77s)
✔ [Éxito]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root -> SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.parquet (385.87s)
✔ [Éxito]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run013.root -> SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run013.parquet (387.57s)
✔ [Éxito]: SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root -> SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.parquet (378.57s)
✔ [Éxito]: SIB23e_175_180_helium_MdSdInfill_C

  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run031.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run012.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run011.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run014.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SIB23e/17.5_18.0/helium/SIB23e_175_180_helium_MdSdInfill_CORSIKA78010_FLUKA_Run032.root
  input file /srv/data/Malargue/icrc2025/test7/IdealMC_CORSIKA/MdSdInfill_CORSIKA78010_FLUKA/SI